In [1]:
!pip install requests

In [2]:
%%writefile common.txt
admin
login
robots.txt
test
backup
uploads
images
css
js
api
dashboard
config

Writing common.txt


In [3]:
import requests
import csv
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin
from requests.exceptions import RequestException

# ==========================================
# CONFIGURATION
# ==========================================

TARGET_URL = "http://10.10.230.135"
WORDLIST = "common.txt"

THREADS = 10
TIMEOUT = 5

# Status code meanings
STATUS_MEANING = {
    200: "OK - Resource exists",
    201: "Created",
    204: "No Content",
    301: "Permanent Redirect",
    302: "Temporary Redirect",
    307: "Temporary Redirect",
    308: "Permanent Redirect",
    401: "Unauthorized",
    403: "Forbidden - Resource may exist",
    404: "Not Found",
    405: "Method Not Allowed",
    500: "Internal Server Error",
    503: "Service Unavailable"
}


# ==========================================
# READ WORDLIST
# ==========================================

try:
    with open(WORDLIST, "r", encoding="utf-8") as file:
        directories = {
            line.strip()
            for line in file
            if line.strip()
        }

except FileNotFoundError:
    print(f"[ERROR] Wordlist '{WORDLIST}' not found.")
    exit()


# ==========================================
# CREATE SESSION
# ==========================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Security-Directory-Scanner/1.0"
})


# ==========================================
# SCAN FUNCTION
# ==========================================

def scan_directory(directory):

    url = urljoin(TARGET_URL.rstrip("/") + "/", directory)

    start_time = time.time()

    try:

        response = session.get(
            url,
            timeout=TIMEOUT,
            allow_redirects=False
        )

        elapsed = round(time.time() - start_time, 3)

        status = response.status_code

        # Determine result type
        if status == 200:
            result_type = "FOUND"

        elif status in [301, 302, 307, 308]:
            result_type = "REDIRECT"

        elif status == 403:
            result_type = "FORBIDDEN"

        elif status == 401:
            result_type = "AUTH REQUIRED"

        elif status == 404:
            result_type = "NOT FOUND"

        elif status >= 500:
            result_type = "SERVER ERROR"

        else:
            result_type = "OTHER"

        return {
            "url": url,
            "status": status,
            "result": result_type,
            "size": len(response.content),
            "time": elapsed,
            "content_type": response.headers.get(
                "Content-Type",
                "Unknown"
            ),
            "location": response.headers.get(
                "Location",
                ""
            )
        }

    except RequestException as error:

        return {
            "url": url,
            "status": "ERROR",
            "result": "CONNECTION ERROR",
            "size": 0,
            "time": 0,
            "content_type": "",
            "location": ""
        }


# ==========================================
# START SCANNER
# ==========================================

print("=" * 70)
print("        ADVANCED WEB DIRECTORY SCANNER")
print("=" * 70)

print(f"Target      : {TARGET_URL}")
print(f"Wordlist    : {WORDLIST}")
print(f"Directories : {len(directories)}")
print(f"Threads     : {THREADS}")
print("=" * 70)

print("\n[+] Starting scan...\n")


results = []

start_scan = time.time()


# ==========================================
# MULTITHREADED SCANNING
# ==========================================

with ThreadPoolExecutor(max_workers=THREADS) as executor:

    futures = [
        executor.submit(scan_directory, directory)
        for directory in directories
    ]

    completed = 0
    total = len(futures)

    for future in as_completed(futures):

        result = future.result()

        results.append(result)

        completed += 1

        # Print only interesting results
        if result["status"] != 404 and result["status"] != "ERROR":

            print(
                f"[{result['status']}] "
                f"{result['result']:15} "
                f"{result['url']} "
                f"({result['size']} bytes, "
                f"{result['time']}s)"
            )

        print(
            f"\rProgress: {completed}/{total}",
            end=""
        )


scan_time = round(time.time() - start_scan, 2)


# ==========================================
# SUMMARY
# ==========================================

print("\n\n" + "=" * 70)
print("                    SCAN SUMMARY")
print("=" * 70)

interesting = [
    r for r in results
    if r["status"] != 404
    and r["status"] != "ERROR"
]

print(f"Total URLs scanned : {len(results)}")
print(f"Interesting URLs   : {len(interesting)}")
print(f"Scan time          : {scan_time} seconds")


# Count status codes

status_counts = {}

for result in results:

    status = result["status"]

    status_counts[status] = status_counts.get(status, 0) + 1


print("\nStatus Code Summary:")

for status, count in sorted(
    status_counts.items(),
    key=lambda x: str(x[0])
):

    meaning = STATUS_MEANING.get(
        status,
        "Unknown"
    )

    print(
        f"  {status}: {count} - {meaning}"
    )


# ==========================================
# SAVE RESULTS TO CSV
# ==========================================

OUTPUT_FILE = "scan_results.csv"

with open(
    OUTPUT_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "url",
            "status",
            "result",
            "size",
            "time",
            "content_type",
            "location"
        ]
    )

    writer.writeheader()

    writer.writerows(results)


print("\n[+] Results saved to:", OUTPUT_FILE)
print("[+] Scan completed!")

        ADVANCED WEB DIRECTORY SCANNER
Target      : http://10.10.230.135
Wordlist    : common.txt
Directories : 12
Threads     : 10

[+] Starting scan...

Progress: 12/12

                    SCAN SUMMARY
Total URLs scanned : 12
Interesting URLs   : 0
Scan time          : 10.02 seconds

Status Code Summary:
  ERROR: 12 - Unknown

[+] Results saved to: scan_results.csv
[+] Scan completed!
